In [11]:
from google.colab import files
import pathlib
import pickle
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                               precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

DATA_PATH = pathlib.Path("/content/revised_master_dts_weekly.csv")
OUT_DIR   = pathlib.Path("/content/hab_outputs_lr_weekly")
OUT_DIR.mkdir(exist_ok=True)

TARGET_SOURCE   = "bloom_or_documented"
TRAIN_END_YEAR  = 2023
MAX_ITER        = 5000
N_CV_SPLITS     = 4
CV_TEST_SIZE    = 46

In [12]:
df = pd.read_csv(DATA_PATH)
print(f"  raw rows: {len(df)}, columns: {len(df.columns)}")

df = df.sort_values(["region", "year", "doy_start"]).reset_index(drop=True)
df = df.dropna(subset=["chlor_a_mean"]).copy()


lag_cols = [
      "sst_mean", "sst_min", "sst_max", "sst_std",
      "rainfall_mm_total_mean", "rainfall_mm_max_daily", "rainy_days_mean",
  ]
for c in lag_cols:
    for lag in (1, 2):
        df[f"{c}_lag{lag}"] = df.groupby("region")[c].shift(lag)

roll_cols = ["sst_mean", "sst_std", "rainfall_mm_total_mean", "rainy_days_mean"]
for c in roll_cols:
    df[f"{c}_roll4w"] = df.groupby("region")[c].shift(1).rolling(4).mean().reset_index(0, drop=True)
    df[f"{c}_roll8w"] = df.groupby("region")[c].shift(1).rolling(8).mean().reset_index(0, drop=True)


anom_cols = ["sst_mean", "chlor_a_mean", "rainfall_mm_total_mean"]
for c in anom_cols:
    clim = df.groupby(["region", "doy_start"])[c].transform("mean")
    df[f"{c}_anomaly"] = df[c] - clim


df["rainfall_cum_4w"] = (
      df.groupby("region")["rainfall_mm_total_mean"]
        .shift(1).rolling(4).sum().reset_index(0, drop=True))
df["rainfall_cum_8w"] = (
      df.groupby("region")["rainfall_mm_total_mean"]
        .shift(1).rolling(8).sum().reset_index(0, drop=True))


df["sst_slope_4w"] = (
      df.groupby("region")["sst_mean"].shift(1)
        - df.groupby("region")["sst_mean"].shift(5)) / 4


df["window_sin"] = np.sin(2 * np.pi * df["doy_start"] / 365)
df["window_cos"] = np.cos(2 * np.pi * df["doy_start"] / 365)
df["region_Kerala"]    = (df["region"] == "Kerala").astype(int)
df["region_Karnataka"] = (df["region"] == "Karnataka").astype(int)

FEATURES = (
      lag_cols
      + [f"{c}_lag{l}" for c in lag_cols for l in (1, 2)]
      + [c for c in df.columns if "_roll" in c or "_anomaly" in c
                                or "_cum_" in c or "_slope_" in c]
      + ["window_sin", "window_cos", "region_Kerala", "region_Karnataka"]
      + ["hab_event_documented", "hab_events_last_52w"]
  )
FEATURES = list(dict.fromkeys(FEATURES))
print(f"  total features: {len(FEATURES)}")

  raw rows: 460, columns: 25
  total features: 41


In [13]:
df["bloom_next_week"] = df.groupby("region")[TARGET_SOURCE].shift(-1)
TARGET = "bloom_next_week"

df_model = df.dropna(subset=FEATURES + [TARGET]).copy().sort_values(
      ["year", "doy_start", "region"]).reset_index(drop=True)

train = df_model[df_model["year"] <= TRAIN_END_YEAR]
test  = df_model[df_model["year"] >  TRAIN_END_YEAR]

X_train, y_train = train[FEATURES], train[TARGET].astype(int)
X_test,  y_test  = test[FEATURES],  test[TARGET].astype(int)

print(f"  usable rows: {len(df_model)}")
print(f"  train: {len(X_train)} rows (positive rate {y_train.mean():.3f})")
print(f"  test:  {len(X_test)} rows (positive rate {y_test.mean():.3f})")

  usable rows: 211
  train: 187 rows (positive rate 0.310)
  test:  24 rows (positive rate 0.125)


In [14]:
X_cv = X_train.reset_index(drop=True)
y_cv = y_train.reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS, test_size=CV_TEST_SIZE)

fold_metrics = []
for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_cv), 1):
      X_tr, y_tr = X_cv.iloc[tr_idx], y_cv.iloc[tr_idx]
      X_te, y_te = X_cv.iloc[te_idx], y_cv.iloc[te_idx]

fold_model = Pipeline([
          ("scale", StandardScaler()),
          ("clf",   LogisticRegression(
              max_iter=MAX_ITER, solver="lbfgs", C=1.0,
              class_weight="balanced", random_state=42)),
      ])
fold_model.fit(X_tr, y_tr)
p = fold_model.predict_proba(X_te)[:, 1]
pred = (p >= 0.5).astype(int)

row = {"fold": fold, "train_n": len(tr_idx), "test_n": len(te_idx),
             "accuracy": accuracy_score(y_te, pred),
             "roc_auc":  roc_auc_score(y_te, p) if y_te.nunique() > 1 else float("nan")}
fold_metrics.append(row)
print(f"fold {fold}: train={row['train_n']}, test={row['test_n']}, "
            f"acc={row['accuracy']:.3f}, AUC={row['roc_auc']:.3f}")

cv_df = pd.DataFrame(fold_metrics)
print(f"\n== CV summary ({N_CV_SPLITS} folds) ==")
print(f"  accuracy: {cv_df.accuracy.mean():.3f} ± {cv_df.accuracy.std():.3f}")
print(f"  ROC-AUC:  {cv_df.roc_auc.mean():.3f} ± {cv_df.roc_auc.std():.3f}")

fold 4: train=141, test=46, acc=0.826, AUC=0.916

== CV summary (4 folds) ==
  accuracy: 0.826 ± nan
  ROC-AUC:  0.916 ± nan


In [18]:
model = Pipeline([
      ("scale", StandardScaler()),
      ("clf",   LogisticRegression(
          max_iter=MAX_ITER, solver="lbfgs", C=1.0,
          class_weight="balanced", random_state=42)),
  ])
model.fit(X_train, y_train)
actual_iter = int(model.named_steps["clf"].n_iter_[0])
print(f"converged in {actual_iter} iterations")

proba = model.predict_proba(X_test)[:, 1]
pred  = (proba >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()

metrics = {
      "accuracy":  accuracy_score(y_test, pred),
      "precision": precision_score(y_test, pred, zero_division=0),
      "recall":    recall_score(y_test, pred, zero_division=0),
      "f1":        f1_score(y_test, pred, zero_division=0),
      "roc_auc":   roc_auc_score(y_test, proba),
  }

print("\n=== FINAL HELD-OUT METRICS (7-day forecast, 2024) ===")
for k, v in metrics.items():
    print(f"  {k:>9s}: {v:.4f}")
print(f"  confusion matrix: TN={tn}  FP={fp}  FN={fn}  TP={tp}")

converged in 30 iterations

=== FINAL HELD-OUT METRICS (7-day forecast, 2024) ===
   accuracy: 0.7917
  precision: 0.3750
     recall: 1.0000
         f1: 0.5455
    roc_auc: 0.9524
  confusion matrix: TN=16  FP=5  FN=0  TP=3


In [19]:
with open(OUT_DIR / "LogisticRegression_weekly.pkl", "wb") as f:
      pickle.dump(model, f)

test[["year", "doy_start", "date_start", "region", "chlor_a_mean", TARGET]].assign(
    proba=proba, pred=pred
  ).to_csv(OUT_DIR / "LR_weekly_predictions.csv", index=False)

coefs = model.named_steps["clf"].coef_.ravel()
pd.DataFrame({
      "feature":       FEATURES,
      "signed_weight": coefs,
      "importance":    np.abs(coefs),
  }).sort_values("importance", ascending=False).to_csv(
      OUT_DIR / "LR_weekly_feature_importance.csv", index=False)

cv_df.to_csv(OUT_DIR / "LR_weekly_cv_folds.csv", index=False)

pd.DataFrame([{
      "model": "LogisticRegression_weekly",
      "n_iterations": actual_iter,
      "n_features": len(FEATURES),
      "cv_accuracy_mean": round(cv_df.accuracy.mean(), 4),
      "cv_accuracy_std":  round(cv_df.accuracy.std(),  4),
      "cv_roc_auc_mean":  round(cv_df.roc_auc.mean(),  4),
      "cv_roc_auc_std":   round(cv_df.roc_auc.std(),   4),
      **{f"heldout_{k}": round(v, 4) for k, v in metrics.items()},
      "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
  }]).to_csv(OUT_DIR / "LR_weekly_metrics.csv", index=False)

print("saved:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name}")

import shutil
from google.colab import files
shutil.make_archive("lr_weekly_outputs", "zip", OUT_DIR)
files.download("lr_weekly_outputs.zip")

saved:
  LR_weekly_cv_folds.csv
  LR_weekly_feature_importance.csv
  LR_weekly_metrics.csv
  LR_weekly_predictions.csv
  LogisticRegression_weekly.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>